# Phase 4: Court Judgment Prediction - GraphProp, Partial Fine-tuning

## Task
Court judgment prediction: read a full case and predict the decision as a binary label. Scored with macro F1.

Model: the pre-trained GraphProp model. Each case is cut into chunks, each chunk becomes a vector, and a head combines the chunk vectors into one prediction. Only the top blocks are trained, with smaller learning rates for lower blocks.

## Input
- Phase-4 data (train, dev, test cases); downloaded from Hugging Face if not attached.
- GraphProp checkpoint and tokenizer.

## Flow of the notebook (from input to output)
1. Set the file paths and the training settings.
2. Write the main pipeline cj_core.py: data loading, cutting long cases into chunks, the document model (chunk encoder and head), training, evaluation and submission writing.
3. Write the fine-tuning helpers cj_peft.py (freezing, LoRMA, layer-wise learning rates, memory-saving blocks).
4. Write the GraphProp model code.
5. Load the model, set up partial fine-tuning, and run the pipeline: download the data, tokenize and chunk the cases, train with evaluation on dev, choose which chunks to use and the decision threshold on dev, and predict the test cases.

## Output
- Submission zips with one prediction per test case.
- Summary of dev scores and one zip with all outputs.

In [1]:
# Settings: file paths and training config
CKPT = "/kaggle/input/datasets/shiva091ty/graphporp-p2-final-pt/legal_graphprop_random_final_random_90_10.pt"
TOK = "/kaggle/input/datasets/shiva091ty/tokenizer-grphporp-p2/tokenizer_graphprop.json"
import os, glob
# It takes path (file path) and pats (file patterns) as input, uses the given path, or searches the input folder for the file and returns a file path.
def _find(path, pats):
    if os.path.exists(path):
        return path
    for p in pats:
        h = sorted(x for x in glob.glob("/kaggle/input/**/" + p, recursive=True) if "resume" not in x and "trainable" not in x and os.path.isfile(x))
        if h:
            return h[0]
    return path
CKPT = _find(CKPT, ['*graphprop*.pt', '*graph*prop*.pt', '*graphprop*/**/*.pt'])
TOK = _find(TOK, ['*graphprop*tok*.json', 'tokenizer*graphprop*.json', '*graphprop*/**/*.json'])
print(CKPT)
print(TOK)
for _p in (CKPT, TOK):
    if not os.path.isfile(_p):
        print("missing:", _p)
        print([x for x in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(x)][:80])
        raise FileNotFoundError(_p)
OUT = "/kaggle/working"
NAME = "p4_7_graphprop_partial_unfreeze"
HEAD_LR = 1.5e-3
LR_TOP = 1e-4
LR_DECAY = 0.9
TOP_K = 8
CFG = {'W': 576, 'maxch': 16, 'first': 2, 'k_lo': 3, 'k_hi': 6, 'last_k': 3, 'batch_docs': 16, 'mb': 8, 'epochs': 3, 'class_power': 0.5, 'label_smooth': 0.05, 'tok_mask': 0.05, 'head_dropout': 0.3, 'hid': 384, 'gru': 256, 'warmup': 0.05, 'clip': 1.0, 'wd': 0.01, 'ema': 0.999, 'evals': 12, 'eval_ch': 4, 'max_hours': 11.0, 'train_frac': 0.8, 'stop_frac': 0.9, 'save_every': 900, 'sanity_step': 150, 'init_scale': 4096.0, 'cal_steps': 30, 'cb': 48, 'seed': 0, 'kd_alpha': 0.0, 'kd_T': 2.0, 'dump_train': False, 'dump_ch': 6, 'smoke': 0, 'eval_docs': 1200, 'ramp_steps': 150, 'patience': 5}
SMOKE = 0

/kaggle/input/datasets/shiva091ty/graphporp-p2-final-pt/legal_graphprop_random_final_random_90_10.pt
/kaggle/input/datasets/shiva091ty/tokenizer-grphporp-p2/tokenizer_graphprop.json


In [2]:
%%writefile cj_core.py
# Main judgment-prediction code: data, model, training (saved as cj_core.py)
import os, re, json, glob, math, time, random, copy, zipfile, pickle, hashlib, threading
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

REPO = "Exploration-Lab/CS-781-Capstone"
T0 = time.time()
NPOS = 6

# It takes name and roots (folders to search) as input, searches the input folders for a file and returns a file path (or None if not found).
def find_file(name, roots=("/kaggle/input", "/kaggle/working", ".")):
    hits = []
    for r in roots:
        for h in glob.glob(os.path.join(r, "**", name), recursive=True):
            low = h.lower()
            if "__macosx" not in low and ("phase_4" in low or "phase4" in low or "cjpe" in low):
                hits.append(h)
    return sorted(hits, key=len)[0] if hits else None

# It takes dest (download folder) as input, downloads the data if it is not attached and returns a dictionary from file name to path.
def ensure_data(dest="/kaggle/working/phase4_data"):
    names = ["train.json", "dev.json", "test.json"]
    if all(find_file(n) for n in names):
        return {n: find_file(n) for n in names}
    from huggingface_hub import HfApi, hf_hub_download, login
    tok = None
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        tok = os.environ.get("HF_TOKEN")
    if not tok:
        import getpass
        tok = getpass.getpass("HF token: ")
    login(token=tok)
    files = HfApi().list_repo_files(REPO, repo_type="dataset")
    os.makedirs(dest, exist_ok=True)
    out = {}
    for n in names:
        c = [f for f in files if f.lower().endswith("/" + n) and ("phase_4" in f.lower() or "cjpe" in f.lower())]
        if not c:
            raise RuntimeError(f"{n} not found; files: {files[:20]}")
        out[n] = hf_hub_download(REPO, sorted(c, key=len)[0], repo_type="dataset", local_dir=dest)
    return out

LABEL_KEYS = ("label", "labels", "y", "outcome", "decision", "judgment", "judgement", "target")
TEXT_KEYS = ("text", "case_text", "document", "content", "case")

# It takes path (file path) as input, loads documents from a JSON file and returns a list of documents.
def load_docs(path):
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):
        raw = [({**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "text": v}) for k, v in raw.items()]
    out = []
    print("fields of the first record:", list(raw[0].keys()), flush=True)
    for it in raw:
        tk = next(k for k in TEXT_KEYS if k in it)
        t = it[tk]
        t = " ".join(map(str, t)) if isinstance(t, list) else str(t)
        lk = next((k for k in LABEL_KEYS if k in it and it[k] is not None), None)
        y = None if lk is None else int(float(it[lk]))
        out.append(dict(id=it["id"], text=re.sub(r"\s+", " ", t).strip(), y=y))
    return out

_TK = None

_DT = np.int32

# It takes text as input, tokenizes one text and returns an array of token ids.
def _tok_one(text):
    return np.asarray(_TK.encode(text), dtype=_DT)

class Doc:
    __slots__ = ("id", "y", "head", "tail", "n", "segs", "pos", "soft")

    # It takes i (index) as input, returns the tokens of one chunk.
    def chunk(self, i):
        a, s, e = self.segs[i]
        return (self.head if a == 0 else self.tail)[s:e]

# It takes id_ (document id), y (labels), ids (token ids), W (chunk size), maxch (max chunks), first (number of first chunks) and eos (end token id) as input, cuts a long document into chunks and returns a Doc object with chunks.
def build_doc(id_, y, ids, W, maxch, first, eos):
    ids = ids if len(ids) else np.asarray([eos], _DT)
    n = len(ids)
    if n <= maxch * W:
        tail, head = ids, ids[:0]
    else:
        tail, head = ids[-(maxch - first) * W:], ids[:first * W]
    d = Doc(); d.id, d.y, d.head, d.tail, d.n, d.soft = id_, y, head, tail, n, None
    segs, bounds = [], []
    for s in range(0, len(head), W):
        e = min(s + W, len(head)); segs.append((0, s, e)); bounds.append((s, e))
    t = []
    e = len(tail)
    while e > 0:
        s = max(0, e - W); t.append((1, s, e)); e = s
    off = n - len(tail)
    for a, s, e in t[::-1]:
        segs.append((a, s, e)); bounds.append((off + s, off + e))
    m = len(segs)
    pos = np.zeros((m, NPOS), np.float32)
    for i, (s, e) in enumerate(bounds):
        k = m - 1 - i
        pos[i] = (s / n, (n - e) / n, math.log1p(k) / 3, math.log1p(n / W) / 3, float(k == 0), float(i == 0))
    d.segs, d.pos = segs, pos
    return d

# It takes raw (raw documents), tok (tokenizer), cfg (settings) and cache (cache file path) as input, tokenizes all documents, using a cache and returns a list of Doc objects.
def tokenize_docs(raw, tok, cfg, cache):
    global _TK, _DT
    _DT = np.uint16 if getattr(tok, "vocab_size", 10**6) < 65535 else np.int32
    key = hashlib.md5(json.dumps([len(raw), cfg["W"], cfg["maxch"], cfg["first"], raw[0]["id"], raw[-1]["id"], len(raw[0]["text"]), len(raw[-1]["text"])], default=str).encode()).hexdigest()[:10]
    path = f"{cache}.{key}.pkl"
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    _TK = tok
    texts = [r["text"] for r in raw]
    t0 = time.time()
    try:
        import multiprocessing as mp
        with mp.get_context("fork").Pool(min(4, os.cpu_count() or 1)) as pool:
            ids = pool.map(_tok_one, texts, chunksize=32)
    except Exception:
        ids = [_tok_one(t) for t in texts]
    docs = [build_doc(r["id"], r["y"], a, cfg["W"], cfg["maxch"], cfg["first"], tok.eos_id) for r, a in zip(raw, ids)]
    print(f"tokenised {len(docs)} documents in {time.time() - t0:.0f}s, mean {np.mean([d.n for d in docs]):.0f} tokens", flush=True)
    pickle.dump(docs, open(path, "wb"))
    return docs

# It takes items (list of items), pad (padding id), dev (device), tok_mask (token masking rate) and rng (random generator) as input, pads chunks into one batch and returns a dictionary of tensors.
def collate(items, pad, dev, tok_mask=0.0, rng=None):
    arrs, bi, ki = [], [], []
    for b, (d, ix) in enumerate(items):
        for j, i in enumerate(ix):
            arrs.append(d.chunk(i)); bi.append(b); ki.append(j)
    N, T = len(arrs), max(len(a) for a in arrs)
    ids = np.full((N, T), pad, np.int64); lens = np.empty(N, np.int64)
    for r, a in enumerate(arrs):
        ids[r, :len(a)] = a; lens[r] = len(a)
    if tok_mask > 0 and rng is not None:
        valid = np.arange(T)[None] < lens[:, None]
        src = np.minimum((rng.random(ids.shape) * lens[:, None]).astype(np.int64), T - 1)
        ids = np.where((rng.random(ids.shape) < tok_mask) & valid, np.take_along_axis(ids, src, 1), ids)
    B, km = len(items), max(len(ix) for _, ix in items)
    pos = np.zeros((B, km, NPOS), np.float32); cm = np.zeros((B, km), bool)
    for b, (d, ix) in enumerate(items):
        pos[b, :len(ix)] = d.pos[ix]; cm[b, :len(ix)] = True
    t = lambda a: torch.from_numpy(a).to(dev)
    return dict(ids=t(ids), lens=t(lens), bi=t(np.array(bi)), ki=t(np.array(ki)), pos=t(pos), cm=t(cm))

class Enc(nn.Module):
    # It takes bb (pre-trained backbone) as input, wraps the backbone as a chunk encoder and returns nothing.
    def __init__(self, bb):
        super().__init__()
        self.m, self.bb = bb.model, bb

    # It takes ids (chunk token ids) and lens (chunk lengths) as input, turns each chunk into one vector and returns one vector per chunk.
    def forward(self, ids, lens):
        sh = copy.copy(self.bb); sh.model = self.m
        h = sh.hidden(ids).float()
        T = h.size(1)
        mask = (torch.arange(T, device=h.device)[None] < lens[:, None]).float()
        mean = (h * mask[..., None]).sum(1) / lens[:, None].clamp(min=1)
        last = h[torch.arange(h.size(0), device=h.device), (lens - 1).clamp(min=0)]
        return torch.cat([mean, last], -1)

class Head(nn.Module):
    # It takes din (decoder input), hid (hidden size), gru (use a GRU) and drop (dropout) as input, creates the head that combines chunk vectors into a prediction and returns nothing.
    def __init__(self, din, hid=384, gru=256, drop=0.3):
        super().__init__()
        self.norm = nn.LayerNorm(din)
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(din, hid), nn.GELU())
        self.pos = nn.Sequential(nn.Linear(NPOS, 32), nn.GELU())
        self.gru = nn.GRU(hid + 32, gru, batch_first=True, bidirectional=True)
        self.att = nn.Linear(2 * gru, 1)
        self.out = nn.Sequential(nn.Dropout(drop), nn.Linear(4 * gru, 1))

    # It takes x (input tensor), pos (chunk positions), cm (chunk mask) and att (also return attention weights) as input, combines chunk vectors and returns the prediction scores.
    def forward(self, x, pos, cm, att=False):
        z = torch.cat([self.proj(self.norm(x)), self.pos(pos)], -1)
        lens = cm.sum(1).clamp(min=1).cpu()
        y, _ = self.gru(nn.utils.rnn.pack_padded_sequence(z, lens, batch_first=True, enforce_sorted=False))
        y, _ = nn.utils.rnn.pad_packed_sequence(y, batch_first=True, total_length=cm.size(1))
        a = self.att(y).squeeze(-1).masked_fill(~cm, -1e4).softmax(-1)
        pooled = (a.unsqueeze(-1) * y).sum(1)
        last = y[torch.arange(y.size(0), device=y.device), (cm.sum(1) - 1).clamp(min=0)]
        lg = self.out(torch.cat([pooled, last], -1)).squeeze(-1)
        return (lg, a) if att else lg

class DocModel(nn.Module):
    # It takes bb (pre-trained backbone) and cfg (settings) as input, joins the chunk encoder and the head into one model and returns nothing.
    def __init__(self, bb, cfg):
        super().__init__()
        self.enc, self.head = Enc(bb), Head(2 * bb.d_model, cfg["hid"], cfg["gru"], cfg["head_dropout"])
        self.dp = None

    # It takes ids (chunk token ids) and lens (chunk lengths) as input, gets chunk vectors and returns chunk vectors.
    def vec(self, ids, lens):
        return (self.dp or self.enc)(ids, lens)

    # It takes b (batch dictionary) as input, runs the whole model on a batch of documents and returns prediction scores.
    def forward(self, b):
        v = self.vec(b["ids"], b["lens"])
        x = v.new_zeros(b["cm"].shape[0], b["cm"].shape[1], v.size(-1))
        x[b["bi"], b["ki"]] = v
        return self.head(x, b["pos"], b["cm"])

class Paused(Exception):
    pass

# It takes e (error) as input, checks whether an error is a GPU out-of-memory error and returns True or False.
def is_oom(e):
    return "out of memory" in str(e).lower()

# It takes m (number of chunks) as input, returns a dictionary from mode name to chunk indices; it is the different ways of choosing which chunks to use.
def sel_modes(m):
    return {"all": list(range(m)), "last1": list(range(m - 1, m)), "last2": list(range(max(0, m - 2), m)), "last4": list(range(max(0, m - 4), m)),
            "last8": list(range(max(0, m - 8), m)), "first1": [0], "first2": list(range(min(2, m))), "first4": list(range(min(4, m))),
            "middle": list(range(2, m - 2)) if m > 4 else list(range(m)), "no_last": list(range(max(1, m - 1))), "no_last2": list(range(max(1, m - 2)))}

# It takes model, docs (documents), dev (device), pad (padding id), take (chunk chooser), cb (chunks per batch) and log (print function) as input, computes chunk vectors for all documents and returns chunk vectors per document.
@torch.no_grad()
def encode_vectors(model, docs, dev, pad, take, cb=32, log=None):
    model.eval()
    items = []
    for di, d in enumerate(docs):
        m = len(d.segs)
        for i in take(m):
            items.append((di, i, len(d.chunk(i))))
    order = sorted(range(len(items)), key=lambda k: items[k][2])
    V = [dict() for _ in docs]
    p, cb0 = 0, cb
    while p < len(order):
        part = [items[k] for k in order[p:p + cb]]
        arrs = [docs[di].chunk(i) for di, i, _ in part]
        T = max(len(a) for a in arrs)
        ids = np.full((len(arrs), T), pad, np.int64); lens = np.empty(len(arrs), np.int64)
        for r, a in enumerate(arrs):
            ids[r, :len(a)] = a; lens[r] = len(a)
        try:
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev == "cuda"):
                v = model.vec(torch.from_numpy(ids).to(dev), torch.from_numpy(lens).to(dev)).float().cpu().numpy()
        except RuntimeError as e:
            if is_oom(e) and cb > 1:
                torch.cuda.empty_cache(); cb = max(1, cb // 2); continue
            raise
        for r, (di, i, _) in enumerate(part):
            V[di][i] = v[r].astype(np.float16)
        p += len(part)
    return V

# It takes model, docs (documents), V (chunk vectors), dev (device), sel (chunk chooser), att (also return attention weights) and bs (batch size) as input, gets prediction probabilities from the head and returns (scores per document, attention info or None).
@torch.no_grad()
def head_probs(model, docs, V, dev, sel, att=False, bs=256):
    model.eval()
    lg_all, att_acc = np.zeros(len(docs), np.float32), {}
    for s in range(0, len(docs), bs):
        part = list(range(s, min(len(docs), s + bs)))
        ixs = []
        for di in part:
            avail = sorted(V[di])
            pick = [i for i in sel(len(docs[di].segs)) if i in V[di]] or avail
            ixs.append(pick)
        km, D = max(len(x) for x in ixs), len(next(iter(V[part[0]].values())))
        x = np.zeros((len(part), km, D), np.float32); pos = np.zeros((len(part), km, NPOS), np.float32); cm = np.zeros((len(part), km), bool)
        for r, di in enumerate(part):
            for j, i in enumerate(ixs[r]):
                x[r, j] = V[di][i]; pos[r, j] = docs[di].pos[i]; cm[r, j] = True
        t = lambda a: torch.from_numpy(a).to(dev)
        out = model.head(t(x), t(pos), t(cm), att=att)
        lg = out[0] if att else out
        lg_all[s:s + len(part)] = lg.float().cpu().numpy()
        if att:
            a = out[1].cpu().numpy()
            for r, di in enumerate(part):
                m = len(ixs[r])
                for j in range(m):
                    att_acc.setdefault(m - 1 - j, []).append(float(a[r, j]))
    return lg_all, {k: float(np.mean(v)) for k, v in att_acc.items()} if att else None

# It takes z (values) as input, returns values between 0 and 1; it is the sigmoid function.
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# It takes y (labels), p (probabilities) and thr (threshold) as input, computes macro F1, precision, recall and accuracy and returns a dictionary of scores.
def metrics(y, p, thr=0.5):
    y = np.asarray(y); pr = (np.asarray(p) >= thr).astype(int)
    cm = np.zeros((2, 2), int); np.add.at(cm, (y, pr), 1)
    tp = np.diag(cm).astype(float); P = tp / np.maximum(cm.sum(0), 1); R = tp / np.maximum(cm.sum(1), 1)
    F = 2 * P * R / np.maximum(P + R, 1e-12)
    return dict(macro_f1=float(F.mean()), macro_p=float(P.mean()), macro_r=float(R.mean()), acc=float(tp.sum() / cm.sum()), f1=F.tolist(), p=P.tolist(), r=R.tolist(), cm=cm.tolist(), thr=float(thr))

# It takes y (labels), p (probabilities), lo (lowest threshold), hi (highest threshold), step and smooth (smoothing width) as input, finds the decision threshold with the best F1 and returns the best threshold.
def best_threshold(y, p, lo=0.05, hi=0.95, step=0.005, smooth=4):
    ts = np.arange(lo, hi + 1e-9, step)
    f = np.array([metrics(y, p, t)["macro_f1"] for t in ts])
    k = np.convolve(np.pad(f, smooth, mode="edge"), np.ones(2 * smooth + 1) / (2 * smooth + 1), mode="valid")
    return float(ts[int(k.argmax())])

class EMA:
    # It takes params (parameters) and decay (decay factor) as input, starts a moving-average copy of the weights and returns nothing.
    def __init__(self, params, decay):
        self.p, self.d, self.n = params, decay, 0
        self.e = [p.detach().clone().float() for p in params]
        self.bak = None

    # It takes nothing (uses the stored weights) as input, updates the moving-average weights and returns nothing.
    def update(self):
        self.n += 1
        d = min(self.d, (1 + self.n) / (10 + self.n))
        torch._foreach_mul_(self.e, d)
        torch._foreach_add_(self.e, [p.detach().float() for p in self.p], alpha=1 - d)

    # It uses the moving-average weights for evaluation and returns nothing.
    def swap_in(self):
        self.bak = [p.detach().clone() for p in self.p]
        for p, e in zip(self.p, self.e):
            p.data.copy_(e.to(p.dtype))

    # It puts the normal weights back and returns nothing.
    def restore(self):
        for p, b in zip(self.p, self.bak):
            p.data.copy_(b)
        self.bak = None

# It takes s (seconds) as input, formats seconds as hours:minutes:seconds and returns time as text.
def hms(s):
    s = int(max(0, s)); return f"{s // 3600}h{(s % 3600) // 60:02d}m"

# It takes model and params (parameters) as input, copies the trainable weights and returns a dictionary of trainable weights.
def tstate(model, params):
    ids = {id(p) for p in params}
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if id(p) in ids}

# It takes model and st (saved weights) as input, loads saved trainable weights and returns nothing.
def load_tstate(model, st):
    sd = dict(model.named_parameters())
    for n, v in st.items():
        sd[n].data.copy_(v.to(sd[n].dtype))

# It takes step, warm (warm-up steps) and total (total steps) as input, returns a learning-rate multiplier; it is the learning-rate multiplier: warm-up then cosine decay.
def lr_factor(step, warm, total):
    return (step + 1) / warm if step < warm else max(0.02, 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm)))))

# It takes model, tr (training data), dev_docs (dev documents), cfg (settings), groups (parameter groups with learning rates), unfreeze (unfreeze schedule), unfreeze_fn (function that unfreezes layers), dev (device), pad (padding id), name, work (work folder) and log (print function) as input, returns the best weights and training history; it is the main training loop with evaluation and saving.
def train(model, tr, dev_docs, cfg, groups, unfreeze, unfreeze_fn, dev, pad, name, work, log=print):
    use_amp = dev == "cuda"
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), weight_decay=cfg["wd"])
    base = [g["lr"] for g in groups]
    scaler = torch.amp.GradScaler("cuda", init_scale=cfg.get("init_scale", 4096.0), enabled=use_amp)
    ema = EMA(params, cfg["ema"])
    ys = np.array([d.y for d in tr]); n1 = max(1, int(ys.sum())); n0 = max(1, len(ys) - n1)
    cw = torch.tensor([(len(ys) / (2 * n0)) ** cfg["class_power"], (len(ys) / (2 * n1)) ** cfg["class_power"]], device=dev)
    log(f"train docs {len(tr)} (positive {n1}, negative {n0}), class weights {cw.tolist()}")
    if torch.cuda.device_count() > 1:
        object.__setattr__(model, "dp", nn.DataParallel(model.enc))
    bsz, E = cfg["batch_docs"], cfg["epochs"]
    spe = math.ceil(len(tr) / bsz)
    total = spe * E
    warm = max(1, int(cfg["warmup"] * total))
    ev_every = max(1, total // cfg["evals"])
    mb = cfg["mb"]
    ck = os.path.join(work, f"{name}.resume.pt")
    cand = [ck] + glob.glob(f"/kaggle/input/**/{name}.resume.pt", recursive=True) + glob.glob(f"/kaggle/input/**/{name}.resume.pt/**/*", recursive=True)
    cand = [c for c in cand if os.path.isfile(c)]
    step, best, best_state, hist, cal_done, used = 0, -1.0, None, [], False, 0.0
    if cand:
        try:
            c = torch.load(cand[0], map_location="cpu", weights_only=False)
            if cfg.get("rescue"):
                best_state, best, step, hist = c["best_state"], c["best"], c["step"], c["hist"]
                raise StopIteration
            load_tstate(model, c["state"]); opt.load_state_dict(c["opt"]); scaler.load_state_dict(c["scaler"])
            for e, s in zip(ema.e, c["ema"]):
                e.copy_(s)
            ema.n = c["ema_n"]; step, best, best_state, hist, total, cal_done = c["step"], c["best"], c["best_state"], c["hist"], c["total"], c["cal"]
            used = c.get("used", 0.0)
            log(f"resumed at step {step}/{total}")
        except StopIteration:
            pass
        except Exception as e:
            log(f"resume failed: {str(e)[:100]}")
            step, best, best_state, hist, cal_done, used = 0, -1.0, None, [], False, 0.0
    ev_every = max(1, total // cfg["evals"])
    if cfg.get("rescue"):
        assert cand and best_state is not None, "rescue: no resume checkpoint with a best state found"
        log(f"rescue: using the best weights (dev {best:.4f}) stored in the checkpoint at step {step}")
        return best_state, best_state, hist, params
    cur_k = -1
    recent, nskip = [], 0
    ev_docs = dev_docs
    if cfg.get("eval_docs") and len(dev_docs) > cfg["eval_docs"]:
        ev_docs = [dev_docs[i] for i in sorted(np.random.default_rng(0).choice(len(dev_docs), cfg["eval_docs"], replace=False))]
    t_sess, step0, t_save, t_ev = time.time(), step, time.time(), 0.0
    stage_t, stage_s, stage_ev, cal_k = t_sess, step0, 0.0, -2
    max_s = cfg["max_hours"] * 3600
    sess_s = cfg.get("session_hours", cfg["max_hours"]) * 3600
    # It uses values from the outer function, unfreezes more layers as training goes on and returns nothing.
    def apply_unfreeze():
        nonlocal cur_k, stage_t, stage_s, stage_ev
        if not unfreeze:
            return
        frac = step / max(1, total)
        k = [kk for f, kk in unfreeze if frac >= f][-1]
        if k != cur_k:
            unfreeze_fn(k); cur_k = k; stage_t, stage_s, stage_ev = time.time(), step, t_ev; log(f"  unfrozen top {k} blocks at step {step}")
    pstd = 0.0
    lr_mult, rollbacks, bad_evals, gn_sum, gn_n, loss_sum, loss_n = 1.0, 0, 0, 0.0, 0, 0.0, 0
    name_of = {id(p): n for n, p in model.named_parameters()}
    # It uses values from the outer function, goes back to the best weights if training gets worse and returns nothing.
    def rollback():
        nonlocal lr_mult, rollbacks, stage_s
        load_tstate(model, best_state)
        for p, e in zip(params, ema.e):
            e.copy_(best_state[name_of[id(p)]].float())
        opt.state.clear(); scaler.update(4096.0 if use_amp else 1.0)
        lr_mult *= cfg.get("rb_decay", 0.5); rollbacks += 1; stage_s = step
        log(f"  rolled back to the best weights (dev {best:.4f}); learning rates now x{lr_mult:g} (rollback {rollbacks})")
    # It uses values from the outer function, quickly scores the model on dev documents and returns the dev score and current weights.
    def quick_eval():
        nonlocal t_ev, pstd
        t = time.time()
        ema.swap_in()
        try:
            V = encode_vectors(model, ev_docs, dev, pad, lambda m: list(range(max(0, m - cfg["eval_ch"]), m)), cfg["cb"])
            lg, _ = head_probs(model, ev_docs, V, dev, lambda m: list(range(m)))
            p = sigmoid(lg); y = [d.y for d in ev_docs]
            f = metrics(y, p, best_threshold(y, p))["macro_f1"]; f5 = metrics(y, p, 0.5)["macro_f1"]
            st = tstate(model, params); pstd = float(np.std(p))
        finally:
            ema.restore()
        t_ev += time.time() - t
        return f, f5, st
    model.train()
    while step < total:
        ep = step // spe
        perm = np.random.default_rng(cfg["seed"] * 100003 + ep).permutation(len(tr))
        s_in = step - ep * spe
        bar = None
        try:
            from tqdm.auto import tqdm
            bar = tqdm(total=total, initial=step, desc="train", mininterval=60, dynamic_ncols=True)
        except Exception:
            pass
        for bi in range(s_in, spe):
            if step >= total:
                break
            apply_unfreeze()
            lr_f = (lr_factor(step, warm, total) * min(1.0, 0.05 + (step - stage_s + 1) / cfg.get("ramp_steps", 1)) if step > warm else lr_factor(step, warm, total)) * lr_mult
            for g, b0 in zip(opt.param_groups, base):
                g["lr"] = b0 * lr_f
            rng = np.random.default_rng((cfg["seed"] * 100003 + ep) * 7919 + bi)
            items, soft = [], []
            for di in perm[bi * bsz:(bi + 1) * bsz]:
                d = tr[di]; m = len(d.segs)
                k = int(rng.integers(cfg["k_lo"], cfg["k_hi"] + 1))
                if m <= k:
                    ix = list(range(m))
                else:
                    lk = min(cfg["last_k"], k)
                    ix = sorted(set(range(m - lk, m)) | set(rng.choice(m - lk, k - lk, replace=False).tolist()))
                items.append((d, ix)); soft.append(np.nan if d.soft is None else d.soft)
            y_all = torch.tensor([d.y for d, _ in items], dtype=torch.float32, device=dev)
            w_all = cw[y_all.long()]; wtot = w_all.sum()
            soft_all = torch.tensor(soft, dtype=torch.float32, device=dev)
            while True:
                try:
                    run = 0.0
                    for s in range(0, len(items), mb):
                        b = collate(items[s:s + mb], pad, dev, cfg["tok_mask"], rng)
                        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                            lg = model(b).float()
                        y, w, zt = y_all[s:s + mb], w_all[s:s + mb], soft_all[s:s + mb]
                        t = y * (1 - cfg["label_smooth"]) + 0.5 * cfg["label_smooth"]
                        ce = F.binary_cross_entropy_with_logits(lg, t, reduction="none")
                        if cfg["kd_alpha"] > 0:
                            has = ~torch.isnan(zt)
                            T = cfg["kd_T"]
                            kd = F.binary_cross_entropy_with_logits(lg / T, torch.sigmoid(torch.nan_to_num(zt) / T), reduction="none") * T * T
                            a = cfg["kd_alpha"] * has.float()
                            ce = (1 - a) * ce + a * kd
                        loss = (w * ce).sum() / wtot
                        scaler.scale(loss).backward(); run += float(loss.detach())
                    break
                except RuntimeError as e:
                    if is_oom(e) and mb > 1:
                        opt.zero_grad(set_to_none=True); torch.cuda.empty_cache(); mb = max(1, mb // 2); log(f"  out of memory -> {mb} documents per micro-batch")
                        continue
                    raise
            scaler.unscale_(opt); gn = float(torch.nn.utils.clip_grad_norm_([p for p in params if p.grad is not None], cfg["clip"]))
            if math.isfinite(gn):
                gn_sum += gn; gn_n += 1
            loss_sum += run; loss_n += 1
            sc0 = scaler.get_scale()
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); ema.update(); step += 1
            skipped = scaler.get_scale() < sc0
            nskip += skipped; recent = (recent + [skipped])[-100:]
            if step == cfg["sanity_step"] and np.mean(recent) > 0.4:
                raise RuntimeError(f"{int(np.sum(recent))} of the last {len(recent)} optimizer steps were skipped (inf/NaN gradients, loss scale {scaler.get_scale():g}): training is not updating the weights")
            if bar is not None:
                bar.update(1)
                if step % 50 == 0:
                    bar.set_postfix(loss=f"{run:.3f}")
            if step % 50 == 0 and bar is None:
                log(f"  step {step}/{total} loss {run:.3f}")
            el = used + time.time() - T0
            if cal_k != cur_k and step - stage_s >= cfg["cal_steps"]:
                cal_k = cur_k; cal_done = True
                sps = (time.time() - stage_t - (t_ev - stage_ev)) / (step - stage_s)
                allow = (cfg["train_frac"] * max_s - el) / (sps * 1.1)
                if step + allow < total:
                    total = max(warm + 50, int(step + allow)); ev_every = max(1, total // cfg["evals"])
                    log(f"  time budget: {sps:.2f}s/step -> total steps reduced to {total}")
                else:
                    log(f"  {sps:.2f}s/step, projected training time {hms((total - step) * sps * 1.1)}")
            stop = el > cfg["stop_frac"] * max_s
            pause = (time.time() - T0 > 0.93 * sess_s) and step < total and not stop
            if step % ev_every == 0 or step >= total or stop:
                model.eval(); f, f5, st = quick_eval(); model.train()
                if f > best:
                    best, best_state = f, st
                tl, gnm = loss_sum / max(1, loss_n), gn_sum / max(1, gn_n); loss_sum, loss_n, gn_sum, gn_n = 0.0, 0, 0.0, 0
                hist.append(dict(step=step, epoch=step / spe, dev_macro_f1_tuned=f, dev_macro_f1_0_5=f5, loss=tl, grad_norm=gnm, lr_mult=lr_mult, skipped_steps=int(nskip), loss_scale=float(scaler.get_scale()), dev_prob_std=pstd))
                left = (total - step) * (time.time() - t_sess - t_ev) / max(1, step - step0)
                log(f"  step {step}/{total} (epoch {step / spe:.2f}) dev macro-F1 tuned {f:.4f} @0.5 {f5:.4f} best {best:.4f} | train loss {tl:.4f} grad norm {gnm:.2f} lr x{lr_mult:g} | prob std {pstd:.3f} skipped {int(nskip)} scale {scaler.get_scale():g} | ETA {hms(left)}")
                bad = (pstd < 0.02 and f < 0.4) or (best > 0 and f < best - cfg.get("drop_tol", 0.05))
                bad_evals = bad_evals + 1 if bad else 0
                if bad and best_state is not None and step < total:
                    if rollbacks < cfg.get("max_rollbacks", 3):
                        rollback()
                    else:
                        log("  dev score keeps falling after repeated rollbacks: stopping and keeping the best weights")
                        total = step
                elif cfg.get("patience") and f <= best and len(hist) >= cfg["patience"] and max(h["dev_macro_f1_tuned"] for h in hist[-cfg["patience"]:]) <= best - 1e-9 and step > 0.6 * total:
                    log("  no dev improvement for several evaluations: stopping"); total = step
            if (time.time() - t_save > cfg["save_every"] and step < total) or (stop and step < total) or pause:
                torch.save(dict(state=tstate(model, params), opt=opt.state_dict(), scaler=scaler.state_dict(), ema=[e.cpu() for e in ema.e], ema_n=ema.n,
                                step=step, best=best, best_state=best_state, hist=hist, total=total, cal=cal_done, used=el), ck + ".tmp")
                os.replace(ck + ".tmp", ck); t_save = time.time()
            if pause:
                raise Paused(f"session time is nearly over: progress saved at step {step}/{total}. Start this notebook again with this run's output attached as an input to continue")
            if stop:
                log("  time limit reached: stopping training"); total = step
                break
        if bar is not None:
            bar.close()
    ema.swap_in()
    last_state = tstate(model, params)
    ema.restore()
    if best_state is None:
        best_state = last_state
    if os.path.exists(ck):
        os.remove(ck)
    return best_state, last_state, hist, params

# It takes model, docs (documents), dev (device), pad (padding id), ch (chunk ids), cb (chunks per batch), path (file path) and log (print function) as input, computes and saves the teacher's predictions and returns teacher scores per document.
def teacher_logits(model, docs, dev, pad, ch, cb, path, log=print):
    done = {}
    if os.path.exists(path):
        done = json.load(open(path))
    todo = [i for i in range(len(docs)) if str(i) not in done]
    for s in range(0, len(todo), 1500):
        idx = todo[s:s + 1500]; part = [docs[i] for i in idx]
        V = encode_vectors(model, part, dev, pad, lambda m: list(range(max(0, m - ch), m)), cb)
        lg, _ = head_probs(model, part, V, dev, lambda m: list(range(m)))
        done.update({str(i): float(z) for i, z in zip(idx, lg)})
        json.dump(done, open(path, "w")); log(f"  teacher logits {len(done)}/{len(docs)}")
    return done

# It takes path (file path) and preds (predictions) as input, writes predictions into a submission zip and returns nothing.
def write_zip(path, preds):
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json", json.dumps(preds))
    with zipfile.ZipFile(path) as z:
        assert z.namelist() == ["prediction.json"]

# It takes ids (token ids), p (probabilities) and thr (threshold) as input, formats predictions for submission and returns a list of predictions in submission format.
def make_preds(ids, p, thr):
    assert len(ids) == len(set(map(str, ids))) == len(p)
    return [{"id": i, "pred": int(x >= thr)} for i, x in zip(ids, p)]

# It takes name, A (model code module), ckpt (checkpoint path), tokf (tokenizer file path), build (function that builds the model), cfg (settings), out (output folder), work (work folder), teacher_dir (teacher output folder) and log (print function) as input, runs the whole pipeline: data, training, evaluation and submissions and returns a results summary.
def main(name, A, ckpt, tokf, build, cfg, out, work, teacher_dir=None, log=print):
    os.makedirs(out, exist_ok=True); os.makedirs(work, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"]); random.seed(cfg["seed"])
    paths = ensure_data()
    raw = {k: load_docs(paths[k + ".json"]) for k in ("train", "dev", "test")}
    if cfg.get("smoke"):
        raw = {k: v[:cfg["smoke"]] for k, v in raw.items()}
    assert all(r["y"] in (0, 1) for r in raw["train"] + raw["dev"]), "labels must be 0/1"
    th = lambda r: hashlib.md5(" ".join(str(r["text"]).split()).encode()).hexdigest()
    tid, tx = {str(r["id"]) for r in raw["test"]}, {th(r) for r in raw["test"]}
    dtx = {th(r) for r in raw["dev"]}
    log(f"overlap with test: dev by id {sum(str(r['id']) in tid for r in raw['dev'])}, dev by text {sum(th(r) in tx for r in raw['dev'])}, train by id {sum(str(r['id']) in tid for r in raw['train'])}, train by text {sum(th(r) in tx for r in raw['train'])}, train-dev text {sum(th(r) in dtx for r in raw['train'])}")
    keep = [r for r in raw["dev"] if str(r["id"]) not in tid and th(r) not in tx]
    if len(keep) != len(raw["dev"]):
        log(f"removed {len(raw['dev']) - len(keep)} dev documents that are also test cases; {len(keep)} left")
        raw["dev"] = keep
    log({k: len(v) for k, v in raw.items()}, "positive share train", np.mean([r["y"] for r in raw["train"]]), "dev", np.mean([r["y"] for r in raw["dev"]]))
    tok = A.load_tokenizer(tokf); pad = tok.pad_id
    docs = {k: tokenize_docs(raw[k], tok, cfg, os.path.join(work, f"{name}_{k}")) for k in raw}
    bb, groups_fn, uf = build()
    model = DocModel(bb, cfg).to(dev)
    groups = groups_fn(model)
    sched, ufn = uf if uf else (None, None)
    ntr = sum(p.numel() for g in groups for p in g["params"])
    nall = sum(p.numel() for p in model.parameters())
    log(f"{name}: trainable parameters {ntr:,} of {nall:,}")
    if teacher_dir:
        fs = glob.glob(os.path.join(teacher_dir, "**", "*_train_logits.json"), recursive=True)
        if not fs:
            for z in glob.glob(os.path.join(teacher_dir, "**", "*_ALL_outputs.zip"), recursive=True):
                zipfile.ZipFile(z).extractall(os.path.join(work, "teacher")); fs = glob.glob(os.path.join(work, "teacher", "**", "*_train_logits.json"), recursive=True)
        assert fs, "teacher logits not found"
        tl = json.load(open(fs[0])); hit = 0; log(f"teacher file {fs[0]} ({len(fs)} found)")
        for i, d in enumerate(docs["train"]):
            if str(i) in tl:
                d.soft = tl[str(i)]; hit += 1
        log(f"teacher logits for {hit}/{len(docs['train'])} training documents")
    t_train = time.time()
    try:
        best_state, last_state, hist, params = train(model, docs["train"], docs["dev"], cfg, groups, sched, ufn, dev, pad, name, work, log)
    except Paused as e:
        log(str(e))
        return None
    train_secs = time.time() - t_train
    peak = torch.cuda.max_memory_allocated() / 2**30 if dev == "cuda" else 0.0
    summary = dict(name=name, cfg={k: v for k, v in cfg.items() if isinstance(v, (int, float, str, bool, list, tuple, type(None)))}, trainable_params=ntr, total_params=nall,
                   backbone_params=int(bb.n_params), model_mb_fp32=nall * 4 / 2**20, peak_gpu_gb=peak, train_seconds=train_secs, history=hist, kinds={})
    torch.save(best_state, os.path.join(out, f"{name}_best_trainable.pt")); torch.save(last_state, os.path.join(out, f"{name}_last_trainable.pt"))
    ydev = np.array([d.y for d in docs["dev"]])
    probs = {}
    inf_t = {}
    for kind, st in (("best", best_state), ("last", last_state)):
        load_tstate(model, st)
        t = time.time()
        Vd = encode_vectors(model, docs["dev"], dev, pad, lambda m: list(range(m)), cfg["cb"])
        inf_t[kind] = (time.time() - t) / len(docs["dev"])
        Vt = encode_vectors(model, docs["test"], dev, pad, lambda m: list(range(m)), cfg["cb"])
        table = {}
        for mname in sel_modes(10):
            lg, _ = head_probs(model, docs["dev"], Vd, dev, lambda m, mname=mname: sel_modes(m)[mname])
            p = sigmoid(lg); thr = best_threshold(ydev, p)
            table[mname] = dict(tuned=metrics(ydev, p, thr), at_0_5=metrics(ydev, p, 0.5))
        _, att = head_probs(model, docs["dev"], Vd, dev, lambda m: list(range(m)), att=True)
        LD, LT_ = {}, {}
        for mname in ("all", "last8", "last4"):
            f = lambda m, mname=mname: sel_modes(m)[mname]
            LD[mname], _ = head_probs(model, docs["dev"], Vd, dev, f); LT_[mname], _ = head_probs(model, docs["test"], Vt, dev, f)
        LD["tta"], LT_["tta"] = np.mean([LD[k] for k in ("all", "last8", "last4")], 0), np.mean([LT_[k] for k in ("all", "last8", "last4")], 0)
        score = {m: metrics(ydev, sigmoid(LD[m]), best_threshold(ydev, sigmoid(LD[m])))["macro_f1"] for m in LD}
        pick = max(score, key=score.get)
        pd_, pt_ = LD[pick], LT_[pick]
        table["tta"] = dict(tuned=metrics(ydev, sigmoid(LD["tta"]), best_threshold(ydev, sigmoid(LD["tta"]))), at_0_5=metrics(ydev, sigmoid(LD["tta"]), 0.5))
        probs[kind] = (sigmoid(pd_), sigmoid(pt_), pick)
        summary["kinds"][kind] = dict(chosen_mode=pick, section_ablation=table, attention_by_chunks_from_end=att)
        log(f"[{kind}] chosen mode {pick}: dev macro-F1 tuned {table[pick]['tuned']['macro_f1']:.4f} @0.5 {table[pick]['at_0_5']['macro_f1']:.4f}")
        log("   section ablation (tuned macro-F1): " + "  ".join(f"{m} {v['tuned']['macro_f1']:.3f}" for m, v in table.items()))
    probs["avg"] = ((probs["best"][0] + probs["last"][0]) / 2, (probs["best"][1] + probs["last"][1]) / 2, "best+last")
    dev_ids, test_ids = [d.id for d in docs["dev"]], [d.id for d in docs["test"]]
    for kind, (pdv, ptt, pick) in probs.items():
        thr = best_threshold(ydev, pdv)
        m_t, m_5 = metrics(ydev, pdv, thr), metrics(ydev, pdv, 0.5)
        summary["kinds"].setdefault(kind, {}).update(threshold=thr, dev_tuned=m_t, dev_at_0_5=m_5, mode=pick)
        log(f"[{kind}] threshold {thr:.3f} dev macro-F1 {m_t['macro_f1']:.4f} (P {m_t['macro_p']:.3f} R {m_t['macro_r']:.3f}) | at 0.5: {m_5['macro_f1']:.4f}")
        np.savez_compressed(os.path.join(out, f"{name}_preds_{kind}.npz"), dev_ids=np.array(dev_ids, dtype=str), dev_p=pdv, dev_y=ydev, test_ids=np.array(test_ids, dtype=str), test_ids_json=np.array(json.dumps(test_ids)), test_p=ptt, thr=thr)
        write_zip(os.path.join(out, f"{name}_{kind}_submission.zip"), make_preds(test_ids, ptt, thr))
        if kind == "best":
            pred = (pdv >= thr).astype(int); lens = np.array([d.n for d in docs["dev"]])
            rows = ["id,y,prob,pred,tokens,chunks"] + [f"{d.id},{d.y},{p:.4f},{q},{d.n},{len(d.segs)}" for d, p, q in zip(docs["dev"], pdv, pred)]
            open(os.path.join(out, f"{name}_dev_predictions.csv"), "w").write("\n".join(rows))
            grp = {g: lens[(ydev == a) & (pred == b)] for g, (a, b) in dict(TP=(1, 1), TN=(0, 0), FP=(0, 1), FN=(1, 0)).items()}
            summary["error_analysis_tokens"] = {g: dict(n=int(len(v)), mean_tokens=float(v.mean()) if len(v) else 0.0, median_tokens=float(np.median(v)) if len(v) else 0.0) for g, v in grp.items()}
    summary["inference_seconds_per_doc"] = inf_t
    summary["gpu_count"] = torch.cuda.device_count()
    if cfg["dump_train"]:
        load_tstate(model, best_state)
        teacher_logits(model, docs["train"], dev, pad, cfg["dump_ch"], cfg["cb"], os.path.join(out, f"{name}_train_logits.json"), log)
    json.dump(summary, open(os.path.join(out, f"{name}_summary.json"), "w"), indent=1)
    for f in glob.glob(os.path.join(work, "*.pkl")):
        os.remove(f)
    zp = os.path.join(out, f"{name}_ALL_outputs.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(glob.glob(os.path.join(out, "*"))):
            if os.path.isfile(f) and not f.endswith("_ALL_outputs.zip") and "phase4_data" not in f:
                z.write(f, os.path.basename(f))
    log(zp, f"{os.path.getsize(zp) / 1e6:.1f} MB", "total time", hms(time.time() - T0))
    return summary

# It takes model, head_lr (head learning rate), peft_lr (learning rate for the adapter weights) and wd (weight decay) as input, sets learning rates for the head and the backbone and returns a list of parameter groups with learning rates.
def default_groups(model, head_lr, peft_lr, wd=0.01):
    bbp = [p for p in model.enc.parameters() if p.requires_grad]
    g = [dict(params=list(model.head.parameters()), lr=head_lr, weight_decay=wd)]
    if bbp:
        g.append(dict(params=bbp, lr=peft_lr, weight_decay=wd))
    return g

Writing cj_core.py


In [3]:
%%writefile cj_peft.py
# Fine-tuning methods: LoRMA, layer-wise learning rates (saved as cj_peft.py)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

# It takes bb (pre-trained backbone) as input, returns the list of model blocks.
def get_blocks(bb):
    return list(bb.block_modules()) if hasattr(bb, "block_modules") else list(bb.model.blocks)

# It takes model as input, freezes all weights of the model and returns nothing.
def freeze(model):
    for p in model.parameters():
        p.requires_grad_(False)

class LoRMALinear(nn.Module):
    # It takes base (base linear layer), r (rank) and alpha (scale of the low-rank update) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.s * F.linear(F.linear(y0, self.A.to(x.dtype)), self.B.to(x.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(x.dtype)

# It takes bb (pre-trained backbone), names (layer names), r (rank) and alpha (scale of the low-rank update) as input, freezes the model and adds LoRMA to the chosen linear layers and returns a number of layers wrapped.
def apply_lorma(bb, names, r=8, alpha=16):
    freeze(bb.model)
    n = 0
    for name in names:
        parent = bb.model
        *path, leaf = name.split(".")
        for p in path:
            parent = getattr(parent, p)
        base = getattr(parent, leaf)
        if isinstance(base, nn.Linear):
            setattr(parent, leaf, LoRMALinear(base, r, alpha))
            n += 1
    assert n > 0, "no Linear layers wrapped"
    return n

# It takes name as input, checks whether a layer is the final norm layer and returns True or False.
def is_final_norm(name):
    return "norm_f" in name or name.startswith("norm.") or name.startswith("ln_f")

# It takes bb (pre-trained backbone), lr_top (top-layer learning rate), decay (decay factor), head_params (head weights), head_lr (head learning rate) and wd (weight decay) as input, gives lower layers a smaller learning rate than top layers and returns a list of parameter groups with learning rates.
def layerwise_lr_groups(bb, lr_top, decay=0.85, head_params=(), head_lr=1e-3, wd=0.01):
    blocks = get_blocks(bb)
    n, seen, groups = len(blocks), set(), []
    for i, blk in enumerate(blocks):
        ps = [p for p in blk.parameters() if p.requires_grad]
        seen.update(id(p) for p in ps)
        groups.append(dict(params=ps, lr=lr_top * decay ** (n - 1 - i), weight_decay=wd))
    rest = [p for p in bb.model.parameters() if p.requires_grad and id(p) not in seen]
    groups.append(dict(params=rest, lr=lr_top * decay ** n, weight_decay=wd))
    groups.append(dict(params=list(head_params), lr=head_lr, weight_decay=wd))
    return [g for g in groups if g["params"]]

# It takes bb (pre-trained backbone), top_k (top-k), lr_top (top-layer learning rate), decay (decay factor), head_params (head weights), head_lr (head learning rate) and wd (weight decay) as input, trains only the top layers, with smaller learning rates lower down and returns a list of parameter groups with learning rates.
def partial_groups(bb, top_k, lr_top, decay, head_params, head_lr, wd=0.01):
    freeze(bb.model)
    groups = [dict(params=list(blk.parameters()), lr=lr_top * decay ** i, weight_decay=wd) for i, blk in enumerate(reversed(get_blocks(bb)[-top_k:]))]
    norm = [p for n, p in bb.model.named_parameters() if is_final_norm(n)]
    for p in norm:
        p.requires_grad_(True)
    if norm:
        groups.append(dict(params=norm, lr=lr_top, weight_decay=0.0))
    groups.append(dict(params=list(head_params), lr=head_lr, weight_decay=wd))
    return groups

# It takes bb (pre-trained backbone) and k (number of top blocks) as input, unfreezes only the top k blocks and returns nothing.
def set_unfrozen(bb, k):
    for j, blk in enumerate(reversed(get_blocks(bb))):
        for p in blk.parameters():
            p.requires_grad_(j < k)

class CkptBlock(nn.Module):
    # It takes blk (model block) as input, wraps a block to save GPU memory and returns nothing.
    def __init__(self, blk):
        super().__init__()
        self.blk = blk

    # It returns the attention layer of the wrapped block.
    @property
    def attn(self):
        return self.blk.attn

    # It takes *a (any arguments) as input, runs the block while saving memory and returns block output.
    def forward(self, *a):
        if self.training and torch.is_grad_enabled():
            return torch.utils.checkpoint.checkpoint(self.blk, *a, use_reentrant=False)
        return self.blk(*a)

# It takes bb (pre-trained backbone) as input, wraps all blocks to save GPU memory and returns nothing.
def checkpoint_blocks(bb):
    for i, blk in enumerate(bb.model.blocks):
        bb.model.blocks[i] = CkptBlock(blk)

Writing cj_peft.py


In [4]:
%%writefile arch_graphprop.py
# GraphProp model code (saved as arch_graphprop.py)
import os, json, zipfile, tempfile, shutil, re as _re
from collections import Counter, defaultdict as _defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

ARCH = "graphprop"

class TokenPosEmbed(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size) and max_len (max length) as input, creates the token and position embedding tables and returns nothing.
    def __init__(self, vocab_size, dim, max_len=1024):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, dim)
        self.pos = nn.Embedding(max_len, dim)

    # It takes input_ids (token ids) and pos_offset (position offset) as input, adds token and position embeddings for the input and returns embedding vectors.
    def forward(self, input_ids, pos_offset=0):
        B, T = input_ids.shape
        positions = (torch.arange(T, device=input_ids.device) + pos_offset).unsqueeze(0).expand(B, -1)
        return self.tok(input_ids) + self.pos(positions)

class FeedForward(nn.Module):

    # It takes dim (model size), mult (size multiplier) and dropout (dropout rate) as input, creates the feed-forward layer and returns nothing.
    def __init__(self, dim, mult=4, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * mult, dim),
            nn.Dropout(dropout),
        )

    # It takes x (input tensor) as input, runs the feed-forward layer and returns output vectors.
    def forward(self, x):
        return x + self.net(self.norm(x))

class GraphPropLayer(nn.Module):

    # It takes dim (model size), window (window size) and edge_dim (edge size) as input, creates the graph message-passing layer and returns nothing.
    def __init__(self, dim, window=16, edge_dim=64):
        super().__init__()
        self.window = window
        self.norm = nn.LayerNorm(dim)
        self.edge_key = nn.Linear(dim, edge_dim)
        self.edge_mlp = nn.Sequential(nn.Linear(edge_dim * 2, edge_dim), nn.GELU(), nn.Linear(edge_dim, 1))
        self.value_proj = nn.Linear(dim, dim)
        self.rel_emb = nn.Embedding(window, dim)
        self.out_proj = nn.Linear(dim, dim)

    # It takes x (input tensor) as input, passes messages from nearby previous tokens to each token and returns output vectors.
    def forward(self, x):

        B, T, D = x.shape
        W = self.window
        h = self.norm(x)

        K = self.edge_key(h)
        e = K.size(-1)
        pad_rows = self.edge_key.bias.to(K.dtype).expand(B, W - 1, -1)
        K_all = torch.cat([pad_rows, K], dim=1)
        w1, b1 = self.edge_mlp[0].weight, self.edge_mlp[0].bias
        A = F.linear(K, w1[:, :e], b1)
        Bn = F.linear(K_all, w1[:, e:])
        z = A.unsqueeze(2) + Bn.unfold(1, W, 1).permute(0, 1, 3, 2)
        edge_w = torch.sigmoid(self.edge_mlp[2](self.edge_mlp[1](z))).squeeze(-1)

        wv, bv = self.value_proj.weight, self.value_proj.bias
        V = F.pad(F.linear(h, wv), (0, 0, W - 1, 0))
        R = F.linear(self.rel_emb.weight, wv, bv)
        t_idx = torch.arange(T, device=x.device).unsqueeze(1) + torch.arange(W, device=x.device).unsqueeze(0)
        M = torch.zeros(B, T, T + W - 1, dtype=edge_w.dtype, device=x.device)
        M.scatter_(2, t_idx.unsqueeze(0).expand(B, -1, -1), edge_w)
        num = torch.bmm(M, V.to(M.dtype)) + torch.matmul(edge_w, R.to(edge_w.dtype))

        denom = edge_w.sum(-1, keepdim=True).clamp_min(1e-6)
        agg = num / denom
        out = self.out_proj(agg)
        return x + out

    # It takes h (normalised hidden vectors), valid (mask of valid neighbours) and drop_self (skip the token itself) as input, computes the message sums used by the layer and returns (message sum, edge weight sum).
    def _num_den(self, h, valid=None, drop_self=False):

        B, T, D = h.shape
        W = self.window
        K = self.edge_key(h)
        e = K.size(-1)
        pad_rows = self.edge_key.bias.to(K.dtype).expand(B, W - 1, -1)
        K_all = torch.cat([pad_rows, K], dim=1)
        w1, b1 = self.edge_mlp[0].weight, self.edge_mlp[0].bias
        A = F.linear(K, w1[:, :e], b1)
        Bn = F.linear(K_all, w1[:, e:])
        z = A.unsqueeze(2) + Bn.unfold(1, W, 1).permute(0, 1, 3, 2)
        edge_w = torch.sigmoid(self.edge_mlp[2](self.edge_mlp[1](z))).squeeze(-1)
        if valid is not None:
            vpad = torch.cat([torch.ones(B, W - 1, dtype=edge_w.dtype, device=h.device),
                              valid.to(edge_w.dtype)], dim=1)
            edge_w = edge_w * vpad.unfold(1, W, 1)
        if drop_self:
            edge_w = torch.cat([edge_w[..., :W - 1], torch.zeros_like(edge_w[..., :1])], dim=-1)
        wv, bv = self.value_proj.weight, self.value_proj.bias
        V = F.pad(F.linear(h, wv), (0, 0, W - 1, 0))
        R = F.linear(self.rel_emb.weight, wv, bv)
        t_idx = torch.arange(T, device=h.device).unsqueeze(1) + torch.arange(W, device=h.device).unsqueeze(0)
        M = torch.zeros(B, T, T + W - 1, dtype=edge_w.dtype, device=h.device)
        M.scatter_(2, t_idx.unsqueeze(0).expand(B, -1, -1), edge_w)
        num = torch.bmm(M, V.to(M.dtype)) + torch.matmul(edge_w, R.to(edge_w.dtype))
        return num, edge_w.sum(-1, keepdim=True)

    # It takes x (input tensor) and valid (mask of valid neighbours) as input, passes messages from tokens on both sides and returns output vectors.
    def forward_bidirectional(self, x, valid=None):

        h = self.norm(x)
        nc, dc = self._num_den(h, valid)
        vr = valid.flip(1) if valid is not None else None
        nr, dr = self._num_den(h.flip(1), vr, drop_self=True)
        num = nc + nr.flip(1)
        den = (dc + dr.flip(1)).clamp_min(1e-6)
        return x + self.out_proj(num / den)

class LegalGraphPropLM(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size), n_layers (number of layers), window (window size), edge_dim (edge size), max_len (max length), dropout (dropout rate) and tie (tie embeddings) as input, builds the full GraphProp language model and returns nothing.
    def __init__(self, vocab_size, dim=512, n_layers=6, window=16, edge_dim=64, max_len=1024,
                 dropout=0.1, tie=True):
        super().__init__()
        self.embed = TokenPosEmbed(vocab_size, dim, max_len)
        self.drop = nn.Dropout(dropout)
        layers = []
        for _ in range(n_layers):
            layers.append(GraphPropLayer(dim, window, edge_dim))
            layers.append(FeedForward(dim, dropout=dropout))
        self.blocks = nn.ModuleList(layers)
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie:
            self.lm_head.weight = self.embed.tok.weight

        self.apply(self._init_weights)
        residual_scale = (2 * n_layers) ** -0.5
        for blk in self.blocks:
            if isinstance(blk, GraphPropLayer):
                nn.init.normal_(blk.out_proj.weight, mean=0.0, std=0.02 * residual_scale)
            elif isinstance(blk, FeedForward):
                nn.init.normal_(blk.net[-2].weight, mean=0.0, std=0.02 * residual_scale)

    # It sets the starting values of the weights and returns nothing.
    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # It takes input_ids (token ids), pos_offset (position offset), targets (target token ids), last_idx (last token positions), pad_id (padding id) and label_smoothing (label smoothing) as input, runs the model and returns next-token scores.
    def forward(self, input_ids, pos_offset=0, targets=None, last_idx=None, pad_id=None, label_smoothing=0.0):
        x = self.drop(self.embed(input_ids, pos_offset=pos_offset))
        for blk in self.blocks:
            x = blk(x)
        h = self.norm_f(x)
        if last_idx is not None:
            rows = torch.arange(h.size(0), device=h.device)
            return self.lm_head(h[rows, last_idx])
        logits = self.lm_head(h)
        if targets is None:
            return logits
        loss_sum = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=pad_id, reduction="sum", label_smoothing=label_smoothing)
        n_tok = (targets != pad_id).sum()
        return loss_sum.reshape(1), n_tok.reshape(1)

# It maps every byte to a printable character for the byte-level tokenizer and returns a dictionary from byte value to character.
def _bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + \
         list(range(ord("\xae"), ord("\xff") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

_BYTE_ENCODER = _bytes_to_unicode()
_BYTE_DECODER = {v: k for k, v in _BYTE_ENCODER.items()}
_PRETOK_RE = _re.compile(r"\'s|\'t|\'re|\'ve|\'m|\'ll|\'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+")

class LegalByteBPETokenizer:

    PAD_TOKEN = "<pad>"
    EOS_TOKEN = "<|endoftext|>"

    # It creates an empty tokenizer and returns nothing.
    def __init__(self):
        self.vocab = {}
        self.id_to_token = {}
        self.merge_rank = {}
        self.pad_token_id = None
        self.eos_token_id = None
        self.padding_side = "right"
        self._cache = {}

    # It returns the vocabulary size.
    def __len__(self):
        return len(self.vocab)

    # It takes texts, target_vocab_size (target vocabulary size), sample_chars_per_doc (characters used per document) and verbose (print progress) as input, learns the BPE merges and vocabulary from the training texts and returns the trained tokenizer.
    def train(self, texts, target_vocab_size, sample_chars_per_doc=None, verbose=True):
        import time as _time
        t0 = _time.time()
        word_freq = Counter()
        for text in texts:
            if sample_chars_per_doc is not None:
                text = text[:sample_chars_per_doc]
            for m in _PRETOK_RE.finditer(text):
                sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
                word_freq[sym] += 1

        self.vocab = {self.PAD_TOKEN: 0, self.EOS_TOKEN: 1}
        for i in range(256):
            self.vocab[_BYTE_ENCODER[i]] = len(self.vocab)
        self.pad_token_id = self.vocab[self.PAD_TOKEN]
        self.eos_token_id = self.vocab[self.EOS_TOKEN]

        words = {i: list(w) for i, w in enumerate(word_freq)}
        freqs = {i: c for i, (w, c) in enumerate(word_freq.items())}
        pair_counts = Counter()
        pair_to_words = _defaultdict(set)
        for wid, syms in words.items():
            f = freqs[wid]
            for i in range(len(syms) - 1):
                p = (syms[i], syms[i + 1])
                pair_counts[p] += f
                pair_to_words[p].add(wid)

        n_merges_needed = target_vocab_size - len(self.vocab)
        n_merges_done = 0
        while n_merges_done < n_merges_needed and pair_counts:
            max_freq = max(pair_counts.values())
            best = min(p for p, c in pair_counts.items() if c == max_freq)
            new_symbol = best[0] + best[1]
            self.merge_rank[best] = n_merges_done
            self.vocab[new_symbol] = len(self.vocab)
            for wid in list(pair_to_words.get(best, ())):
                syms, f = words[wid], freqs[wid]
                for i in range(len(syms) - 1):
                    p = (syms[i], syms[i + 1])
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_to_words[p].discard(wid)
                merged, i = [], 0
                while i < len(syms):
                    if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                        merged.append(new_symbol); i += 2
                    else:
                        merged.append(syms[i]); i += 1
                words[wid] = merged
                for i in range(len(merged) - 1):
                    p = (merged[i], merged[i + 1])
                    pair_counts[p] += f
                    pair_to_words[p].add(wid)
            pair_counts.pop(best, None)
            pair_to_words.pop(best, None)
            n_merges_done += 1

        self.id_to_token = {v: k for k, v in self.vocab.items()}
        if verbose:
            print(f"    BPE training: {n_merges_done} merges learned (target {n_merges_needed}) in "
                  f"{_time.time() - t0:.1f}s over {len(word_freq)} unique pretoken types "
                  f"-> final vocab size = {len(self.vocab)}")
        return self

    # It takes symbol_tuple (word as symbols) as input, applies the learned merges to one word and returns the word split into tokens (as a string).
    def _bpe(self, symbol_tuple):
        cached = self._cache.get(symbol_tuple)
        if cached is not None:
            return cached
        word = list(symbol_tuple)
        while len(word) > 1:
            best_rank, best_i = None, -1
            for i in range(len(word) - 1):
                r = self.merge_rank.get((word[i], word[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_i = r, i
            if best_rank is None:
                break
            word = word[:best_i] + [word[best_i] + word[best_i + 1]] + word[best_i + 2:]
        self._cache[symbol_tuple] = word
        return word

    # It takes text as input, converts text into a list of token ids and returns a list of token ids.
    def encode(self, text):
        ids = []
        for m in _PRETOK_RE.finditer(text):
            sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
            for piece in self._bpe(sym):
                ids.append(self.vocab[piece])
        return ids

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        special = {self.pad_token_id, self.eos_token_id}
        pieces = [self.id_to_token.get(int(i), "") for i in ids
                  if not (skip_special_tokens and int(i) in special)]
        raw = bytes(_BYTE_DECODER[c] for c in "".join(pieces) if c in _BYTE_DECODER)
        return raw.decode("utf-8", errors="replace")

    # It takes path (file path) as input, saves the tokenizer to a JSON file and returns nothing.
    def save(self, path):
        data = {
            "vocab": self.vocab,
            "merges": [list(pair) for pair, _rank in
                       sorted(self.merge_rank.items(), key=lambda kv: kv[1])],
            "pad_token_id": self.pad_token_id,
            "eos_token_id": self.eos_token_id,
            "padding_side": self.padding_side,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    # It takes path (file path) as input, loads a tokenizer from a JSON file and returns a tokenizer.
    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.id_to_token = {int(v): k for k, v in tok.vocab.items()}
        tok.merge_rank = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        tok.pad_token_id = data["pad_token_id"]
        tok.eos_token_id = data["eos_token_id"]
        tok.padding_side = data.get("padding_side", "right")
        return tok

    # It takes text, max_length (max length), truncation (cut long inputs), padding (padding option) and return_tensors (return tensors) as input, tokenizes one text or a list of texts, with optional cutting and padding and returns a dictionary with input_ids and attention_mask.
    def __call__(self, text, max_length=None, truncation=False, padding=None, return_tensors=None):
        is_batch = isinstance(text, (list, tuple))
        texts = list(text) if is_batch else [text]
        all_ids = [self.encode(t) for t in texts]
        if truncation and max_length is not None:
            all_ids = [ids[:max_length] for ids in all_ids]
        lens = [len(ids) for ids in all_ids]
        if padding == "max_length" and max_length is not None:
            pad_to = max_length
        elif padding is True or padding == "longest":
            pad_to = max(lens) if lens else 0
        else:
            pad_to = None
        if pad_to is not None:
            if self.padding_side == "right":
                attn = [[1] * L + [0] * (pad_to - L) for L in lens]
                all_ids = [ids + [self.pad_token_id] * (pad_to - len(ids)) for ids in all_ids]
            else:
                attn = [[0] * (pad_to - L) + [1] * L for L in lens]
                all_ids = [[self.pad_token_id] * (pad_to - len(ids)) + ids for ids in all_ids]
        else:
            attn = [[1] * L for L in lens]
        if return_tensors == "pt":
            return {"input_ids": torch.tensor(all_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attn, dtype=torch.long)}
        return {"input_ids": all_ids if is_batch else all_ids[0],
                "attention_mask": attn if is_batch else attn[0]}

class Tok:

    # It takes inner (inner tokenizer) as input, loads the tokenizer and returns nothing.
    def __init__(self, inner):
        self.inner = inner
        self.vocab_size = len(inner)
        self.pad_id = inner.pad_token_id
        self.eos_id = inner.eos_token_id
        self.bos_id = None

    # It takes text as input, converts text into token ids and returns a list of token ids.
    def encode(self, text):
        return self.inner.encode(text)

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        return self.inner.decode(ids, skip_special_tokens)

    # It takes text as input, converts text into token ids and remember where each token is in the text and returns (token ids, character offsets).
    def encode_with_offsets(self, text):

        inner = self.inner
        ids, offs = [], []
        for m in _PRETOK_RE.finditer(text):
            s = m.group()
            b = s.encode("utf-8")
            byte2char = []
            for k, ch in enumerate(s):
                byte2char.extend([m.start() + k] * len(ch.encode("utf-8")))
            sym = tuple(_BYTE_ENCODER[x] for x in b)
            pos = 0
            for piece in inner._bpe(sym):
                n = len(piece)
                ids.append(inner.vocab[piece])
                offs.append((byte2char[pos], byte2char[pos + n - 1] + 1))
                pos += n
        return ids, offs

# It takes tok_path (tokenizer file path) as input, loads the tokenizer from its JSON file and returns a tokenizer.
def load_tokenizer(tok_path):

    return Tok(LegalByteBPETokenizer.load(tok_path))

# It takes path (file path) as input, turns an unzipped checkpoint folder back into a .pt file and returns (file path, temporary folder or None).
def _materialise_checkpoint_path(path):

    if not os.path.isdir(path):
        return path, None
    pkl = None
    for root, _d, files in os.walk(path):
        if "data.pkl" in files:
            pkl = root
            break
    if pkl is None:
        raise FileNotFoundError(f"{path} is a directory but contains no data.pkl")
    tmpdir = tempfile.mkdtemp(prefix="gp_ckpt_")
    out = os.path.join(tmpdir, "ckpt.pt")
    top = "archive"
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_STORED) as zf:
        for root, _d, files in os.walk(pkl):
            for fn in sorted(files):
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, pkl).replace(os.sep, "/")
                zf.write(full, arcname=f"{top}/{rel}")
    return out, tmpdir

# It takes obj (object) as input, gets the weights dictionary out of a saved checkpoint and returns weights dictionary.
def _extract_state_dict(obj):

    if isinstance(obj, nn.Module):
        return obj.state_dict()
    if isinstance(obj, dict):
        for k in ("best_state", "state_dict", "model_state_dict", "model_state", "model", "weights"):
            v = obj.get(k)
            if isinstance(v, nn.Module):
                return v.state_dict()
            if isinstance(v, dict) and any(torch.is_tensor(t) for t in v.values()):
                return v
        if any(torch.is_tensor(t) for t in obj.values()):
            return obj
    raise ValueError("could not find a state_dict in checkpoint")

# It takes sd (weights dictionary) as input, removes extra name prefixes from weight names and returns weights dictionary with clean names.
def _strip_prefixes(sd):
    out = {}
    for k, v in sd.items():
        while k.startswith("module."):
            k = k[len("module."):]
        out[k] = v
    return out

# It takes sd (weights dictionary) as input, works out the model settings from the saved weights and returns a dictionary of model settings.
def infer_config(sd):
    vocab, dim = sd["embed.tok.weight"].shape
    max_len = sd["embed.pos.weight"].shape[0]
    idx = {int(m.group(1)) for k in sd for m in [_re.match(r"blocks\.(\d+)\.", k)] if m}
    n_blocks = max(idx) + 1
    assert n_blocks % 2 == 0, "blocks should alternate GraphPropLayer/FeedForward"
    window = sd["blocks.0.rel_emb.weight"].shape[0]
    edge_dim = sd["blocks.0.edge_key.weight"].shape[0]
    return dict(vocab_size=vocab, dim=dim, n_layers=n_blocks // 2, window=window,
                edge_dim=edge_dim, max_len=max_len)

class Backbone:

    causal = True

    # It takes model and cfg (settings) as input, wraps the pre-trained model with simple helper methods and returns nothing.
    def __init__(self, model, cfg):
        self.model = model
        self.cfg = cfg
        self.d_model = cfg["dim"]
        self.max_len = cfg["max_len"]
        self.n_params = sum(p.numel() for p in model.parameters())
        self.bidirectional = False

    # It takes flag (on/off) as input, turns the causal mask off or on and returns nothing.
    def set_bidirectional(self, flag=True):
        self.bidirectional = bool(flag)
        self.causal = not self.bidirectional

    # It takes input_ids (token ids), attention_mask (mask of real tokens) and pos_offset (position offset) as input, returns the hidden vector of every token.
    def hidden(self, input_ids, attention_mask=None, pos_offset=0):
        m = self.model
        T = input_ids.size(1)
        assert T + pos_offset <= self.max_len, f"sequence {T}+{pos_offset} exceeds max_len={self.max_len}"
        x = m.drop(m.embed(input_ids, pos_offset=pos_offset))
        if self.bidirectional:
            valid = attention_mask.bool() if attention_mask is not None else None
            for blk in m.blocks:
                x = blk.forward_bidirectional(x, valid) if isinstance(blk, GraphPropLayer) else blk(x)
        else:
            for blk in m.blocks:
                x = blk(x)
        return m.norm_f(x)

    # It takes input_ids (token ids) and pos_offset (position offset) as input, returns next-token scores.
    def logits(self, input_ids, pos_offset=0):
        return self.model(input_ids, pos_offset=pos_offset)

    # It returns the list of model blocks.
    def block_modules(self):

        return list(self.model.blocks)

    # It returns the names of the linear layers inside the blocks.
    def linear_names(self):

        return [f"blocks.{n}" if n else "blocks"
                for n, mod in self.model.blocks.named_modules() if isinstance(mod, nn.Linear)]

# It takes ckpt_path (checkpoint path), device and dropout (dropout rate) as input, loads the pre-trained model from its checkpoint and returns the backbone ready to use.
def load_backbone(ckpt_path, device="cpu", dropout=0.1):

    path, tmp = _materialise_checkpoint_path(ckpt_path)
    try:
        obj = torch.load(path, weights_only=False, map_location="cpu")
    finally:
        if tmp:
            shutil.rmtree(tmp, ignore_errors=True)
    sd = _strip_prefixes(_extract_state_dict(obj))
    cfg = infer_config(sd)
    tied = ("lm_head.weight" not in sd) or torch.equal(sd["lm_head.weight"], sd["embed.tok.weight"])
    model = LegalGraphPropLM(cfg["vocab_size"], dim=cfg["dim"], n_layers=cfg["n_layers"],
                             window=cfg["window"], edge_dim=cfg["edge_dim"], max_len=cfg["max_len"],
                             dropout=dropout, tie=tied)
    if tied and "lm_head.weight" not in sd:
        sd = dict(sd); sd["lm_head.weight"] = sd["embed.tok.weight"]
    res = model.load_state_dict(sd, strict=False)
    if res.missing_keys or res.unexpected_keys:
        print("!" * 70)
        print("!!! STATE-DICT MISMATCH when loading GraphProp checkpoint !!!")
        print("missing:", res.missing_keys)
        print("unexpected:", res.unexpected_keys)
        print("!" * 70)
        raise RuntimeError("GraphProp checkpoint keys do not match model")
    model.to(device).eval()
    cfg["tied"] = tied
    return Backbone(model, cfg)

Writing arch_graphprop.py


In [6]:
# Build the model (partial fine-tuning) and train it
for _m in ("cj_core", "cj_peft", "arch_graphprop"):
    sys.modules.pop(_m, None)
sys.path.insert(0, os.getcwd())
import cj_core as C, cj_peft as P, arch_graphprop as A
WORK = os.path.join(OUT, "_work")
CFG["smoke"] = SMOKE

# It builds the model with the chosen fine-tuning method and its learning rates and returns (model, learning-rate groups, info).
def build():
    bb = A.load_backbone(CKPT, device="cpu")
    return bb, lambda m: P.partial_groups(bb, TOP_K, LR_TOP, LR_DECAY, list(m.head.parameters()), HEAD_LR), ([(0.0, 2), (0.34, 5), (0.67, TOP_K)], lambda k: P.set_unfrozen(bb, k))

summary = C.main(NAME, A, CKPT, TOK, build, CFG, OUT, WORK)

fields of the first record: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
fields of the first record: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
fields of the first record: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
overlap with test: dev by id 0, dev by text 0, train by id 0, train by text 0, train-dev text 0
{'train': 37387, 'dev': 1988, 'test': 1517} positive share train 0.4097681012116511 dev 0.5
p4_7_graphprop_partial_unfreeze: trainable parameters 97,602,982 of 212,407,018
train docs 37387 (positive 15320, negative 22067), class weights [0.9203937649726868, 1.1046277284622192]
resumed at step 2588/7011


train:  37%|###6      | 2588/7011 [00:00<?, ?it/s]

  unfrozen top 5 blocks at step 2588
  1.19s/step, projected training time 1h35m
  step 2920/7011 (epoch 1.25) dev macro-F1 tuned 0.6123 @0.5 0.6062 best 0.6123 | train loss 0.6277 grad norm 2.84 lr x1 | prob std 0.182 skipped 0 scale 1024 | ETA 1h19m
  step 3504/7011 (epoch 1.50) dev macro-F1 tuned 0.5933 @0.5 0.5964 best 0.6123 | train loss 0.6171 grad norm 2.69 lr x1 | prob std 0.160 skipped 1 scale 512 | ETA 1h08m
  step 4088/7011 (epoch 1.75) dev macro-F1 tuned 0.6033 @0.5 0.5980 best 0.6123 | train loss 0.6099 grad norm 2.74 lr x1 | prob std 0.176 skipped 1 scale 512 | ETA 0h57m
  step 4672/7011 (epoch 2.00) dev macro-F1 tuned 0.6073 @0.5 0.6073 best 0.6123 | train loss 0.5973 grad norm 2.56 lr x1 | prob std 0.171 skipped 1 scale 512 | ETA 0h45m


train:  67%|######6   | 4674/7011 [00:00<?, ?it/s]

  unfrozen top 8 blocks at step 4698
  1.36s/step, projected training time 0h56m
  step 5256/7011 (epoch 2.25) dev macro-F1 tuned 0.6060 @0.5 0.6060 best 0.6123 | train loss 0.5627 grad norm 3.23 lr x1 | prob std 0.193 skipped 2 scale 512 | ETA 0h35m
  step 5840/7011 (epoch 2.50) dev macro-F1 tuned 0.6139 @0.5 0.6105 best 0.6139 | train loss 0.5672 grad norm 3.41 lr x1 | prob std 0.202 skipped 3 scale 256 | ETA 0h24m
  step 6424/7011 (epoch 2.75) dev macro-F1 tuned 0.6269 @0.5 0.6119 best 0.6269 | train loss 0.5499 grad norm 3.31 lr x1 | prob std 0.209 skipped 3 scale 256 | ETA 0h12m
  step 7008/7011 (epoch 3.00) dev macro-F1 tuned 0.6217 @0.5 0.6149 best 0.6269 | train loss 0.5493 grad norm 3.35 lr x1 | prob std 0.213 skipped 3 scale 256 | ETA 0h00m
  step 7011/7011 (epoch 3.00) dev macro-F1 tuned 0.6232 @0.5 0.6149 best 0.6269 | train loss 0.6446 grad norm 4.24 lr x1 | prob std 0.213 skipped 3 scale 256 | ETA 0h00m
[best] chosen mode last8: dev macro-F1 tuned 0.6289 @0.5 0.6221
   se